<h2> Imports, loading event-log function and cleaning pipeline </h2>

In [22]:
import numpy as np
import pandas as pd
import pm4py
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder
import statistics
from collections import Counter
from sklearn.preprocessing import OneHotEncoder

In [23]:
# Importing dataset from file path
def import_xes(file_path):
    log = pm4py.read_xes(file_path)
    return pm4py.convert_to_dataframe(log)

# Cleaning dataset: removing unnecessary columns, shifting to resource focus
def clean_dataset(df):
    df_final = df[['case:concept:name', 'concept:name', 'org:resource', 'time:timestamp']]
    df_final = df_final.sort_values(['org:resource', 'time:timestamp'])
    return df_final


# creating 80/20 split based on resources, ensuring a resource is in EITHER the test set OR the train set
def create_split(df_clean, test_size):
    resource_traces = (
        df_clean.sort_values(["org:resource", "time:timestamp"])
               .groupby("org:resource")["concept:name"]
               .apply(list)
    )

    resource_traces = resource_traces[resource_traces.apply(len) > 1]

    resources = resource_traces.index.tolist()

    # create set of train/test resource ids
    train_res, test_res = train_test_split(
        resources,
        test_size=test_size,
        random_state=1
    )

    train_traces = resource_traces.loc[train_res]
    test_traces = resource_traces.loc[test_res]

    return train_traces, test_traces


def build_prefix_df(resource_traces, min_len=1, max_len=None):
    all_rows = []
    for resource, seq in resource_traces.items():
        max_prefix_len = len(seq) if max_len is None else min(max_len, len(seq))

        for k in range(min_len, max_prefix_len):
            prefix = seq[:k]
            next_act = seq[k] if k < len(seq) else None

            all_rows.append({
                'resource': resource,
                'prefix': prefix,
                'prefix_length': k,
                'last_activity': prefix[-1] if len(prefix) else None,
                'next_activity': next_act
            })
    return pd.DataFrame(all_rows)

# bucket on prefix length
def apply_bucketing_prefix_length(df):
    df = df.copy()
    df['bucket_key'] = df.apply(
        lambda row: (row['prefix_length']), axis=1
    )
    df['bucket_id'] = df['bucket_key'].apply(lambda k: abs(hash(k)) % 10_000_000)
    return df


def process_dataset(df, bucket_type='prefix'):
    df_clean = clean_dataset(df)

    train_split, test_split = create_split(df_clean, 0.2)

    train_df = build_prefix_df(train_split)
    test_df = build_prefix_df(test_split)
    if(bucket_type == 'prefix'):
        train_df = apply_bucketing_prefix_length(train_df)
        test_df = apply_bucketing_prefix_length(test_df)

    return train_df, test_df


<h1> Loading event-logs and transforming</h1>

<h4> Loading datasets </h4>

In [ ]:
# df_2013, df_2013_prefixes = process_dataset("datasets/BPI_Challenge_2013_incidents.xes")
df_2013 = import_xes("datasets/BPI_Challenge_2013_incidents.xes")
train_df_2013, test_df_2013 = process_dataset(df_2013)

parsing log, completed traces :: 100%|██████████| 7554/7554 [00:02<00:00, 2880.76it/s]


In [35]:
def train_RF(X_train, X_test, y_train, y_test):
    rf = RandomForestClassifier(n_estimators=100, random_state=1)
    rf.fit(X_train, y_train)

    y_pred = rf.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy is: {accuracy}")
    f1score = f1_score(y_test, y_pred, average='weighted')
    print(f"f1score = {f1score}")

    report = classification_report(y_test, y_pred, output_dict=True)
    print(report)

In [36]:
def apply_global_OHE(df_train, df_test):
    feature_cols = ['last_activity']
    one_hot_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore') # make unknown activities all 0

    # Fitting encoder on train data
    one_hot_encoder.fit(df_train[feature_cols])

    X_train_ohe = one_hot_encoder.transform(df_train[feature_cols])
    X_test_ohe = one_hot_encoder.transform(df_test[feature_cols])

    y_train = df_train['next_activity']
    y_test = df_test['next_activity']

    return X_train_ohe, X_test_ohe, y_train, y_test


def apply_bucketed_OHE(train_df, test_df):
    feature_cols = ['last_activity']
    global_ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

    global_ohe.fit(train_df[feature_cols])

    # Store results
    bucket_results = []
    models = {}

    unique_buckets = sorted(train_df['bucket_key'].unique())
    print(f"Amount of unique buckets: {len(unique_buckets)}")

    for bucket in unique_buckets:
        # filter out data for this bucket
        train_bucket = train_df[train_df['bucket_key'] == bucket]
        test_bucket  = test_df[test_df['bucket_key'] == bucket]

        if len(test_bucket) == 0:
            continue

        X_train = global_ohe.transform(train_bucket[feature_cols])
        y_train = train_bucket['next_activity']
        
        X_test = global_ohe.transform(test_bucket[feature_cols])
        y_test = test_bucket['next_activity']

        rf = RandomForestClassifier(n_estimators=100, random_state=42) 
        rf.fit(X_train, y_train)
        
        models[bucket] = rf
        
        y_pred = rf.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='weighted')
        
        #print(f"Bucket {bucket}: Accuracy = {acc:.4f} (Support: {len(test_bucket)})")
        
        bucket_results.append({
            'bucket': bucket,
            'accuracy': acc,
            'f1score':f1,
            'support': len(test_bucket) # Weighted average needs this
        })

    total_samples = sum(r['support'] for r in bucket_results)
    weighted_acc = sum(r['accuracy'] * r['support'] for r in bucket_results) / total_samples
    weighted_f1 = sum(r['f1score'] * r['support'] for r in bucket_results) / total_samples

    print("-" * 30)
    print(f"Weighted Average Accuracy: {weighted_acc:.4f}")
    print(f"Weighted Average F1score: {weighted_f1:.4f}")




X_train, X_test, y_train, y_test = apply_global_OHE(train_df_2013, test_df_2013)
print("Global OHE performance:")
train_RF(X_train, X_test, y_train, y_test)


print("Agregated average bucketed performance")
apply_bucketed_OHE(train_df_2013, test_df_2013)

Global OHE performance:
Accuracy is: 0.5972413793103448
f1score = 0.5281455854254985
{'Accepted': {'precision': 0.6566966398485565, 'recall': 0.8447583099963473, 'f1-score': 0.7389498349131963, 'support': 8213.0}, 'Completed': {'precision': 0.24034090909090908, 'recall': 0.22135007849293564, 'f1-score': 0.23045491691637157, 'support': 1911.0}, 'Queued': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 2201.0}, 'accuracy': 0.5972413793103448, 'macro avg': {'precision': 0.29901251631315523, 'recall': 0.35536946282976095, 'f1-score': 0.3231349172765226, 'support': 12325.0}, 'weighted avg': {'precision': 0.4748674223406833, 'recall': 0.5972413793103448, 'f1-score': 0.5281455854254985, 'support': 12325.0}}
Agregated average bucketed performance
Amount of unique buckets: 6161


/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no pre

------------------------------
Weighted Average Accuracy: 0.6554
Weighted Average F1score: 0.5344


<h3> Creating last-state encoding </h3>

<h3> Creating bigrams </h3>

In [17]:
def create_bigrams(seq):
    """Return consecutive activity pairs for a sequence."""
    return [(seq[i], seq[i + 1]) for i in range(len(seq) - 1)]


def create_bigram_vocab(df):
    """Build a bigram vocabulary using the prefixes in df."""
    bigram_set = set()
    for prefix in df['prefix']:
        if not prefix:
            continue
        for bigram in create_bigrams(prefix):
            bigram_set.add(bigram)

    bigram_list = sorted(bigram_set)
    bigram2idx = {bg: i for i, bg in enumerate(bigram_list)}
    return bigram2idx, bigram_list


def encode_prefix_bigram(prefix, bigram2idx):
    vec = np.zeros(len(bigram2idx), dtype=float)
    counts = Counter(create_bigrams(prefix))

    for bg, cnt in counts.items():
        if bg in bigram2idx:
            vec[bigram2idx[bg]] = cnt
    return vec


def bigram_encode_dataset(df_prefix, bigram2idx):
    X = []
    y = []

    for _, row in df_prefix.iterrows():
        vec = encode_prefix_bigram(row['prefix'], bigram2idx)
        X.append(vec)
        y.append(row['next_activity'])

    return np.array(X), np.array(y)
